In [1]:
# ── Cell 1: Install ───────────────────────────────────────────────────────────
!pip install sacrebleu bert-score jiwer librosa -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 43.9 MB/s eta 0:00:0000:0100:01


In [3]:
# ── Cell 2: Full Evaluation ───────────────────────────────────────────────────
import os
import torch
import librosa
import pandas as pd
import jiwer
from tqdm import tqdm
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from sacrebleu.metrics import BLEU, CHRF
from bert_score import score as bert_score

# ── Paths ─────────────────────────────────────────────────────────────────────
MODEL_PATH = '/kaggle/input/models/fatimafai/whisper-small-final/other/default/1/whisper_burushaski_final'
AUDIO_DIR  = '/kaggle/input/datasets/fatimafai/test-audios/MEHTAAB/audio'
TEXT_DIR   = '/kaggle/input/datasets/fatimafai/test-audios/MEHTAAB/text'

# ── Load Model ────────────────────────────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

processor = WhisperProcessor.from_pretrained(MODEL_PATH)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_PATH).to(device)
model.eval()
print("Model loaded successfully!")

# ── Match audio & text files by filename stem ─────────────────────────────────
audio_files = sorted([f for f in os.listdir(AUDIO_DIR) if f.endswith('.wav')])
text_files  = sorted([f for f in os.listdir(TEXT_DIR)  if f.endswith('.txt')])

# Build lookup: stem → text file
text_lookup = {os.path.splitext(f)[0]: f for f in text_files}

matched_pairs = []
for audio_file in audio_files:
    stem = os.path.splitext(audio_file)[0]
    if stem in text_lookup:
        matched_pairs.append((audio_file, text_lookup[stem]))

print(f"Matched pairs found: {len(matched_pairs)} / {len(audio_files)}")

# ── Inference ─────────────────────────────────────────────────────────────────
hypotheses = []
references  = []
filenames   = []
failed      = []

for audio_file, text_file in tqdm(matched_pairs, desc="Evaluating"):
    audio_path = os.path.join(AUDIO_DIR, audio_file)
    text_path  = os.path.join(TEXT_DIR,  text_file)

    try:
        # Load reference
        with open(text_path, 'r', encoding='utf-8') as f:
            reference = f.read().strip()

        # Load audio at 16kHz (Whisper requirement)
        audio, _ = librosa.load(audio_path, sr=16000)

        # Process & generate
        inputs = processor(
            audio,
            sampling_rate=16000,
            return_tensors='pt'
        ).input_features.to(device)

        with torch.no_grad():
            predicted_ids = model.generate(inputs)

        hypothesis = processor.batch_decode(
            predicted_ids, skip_special_tokens=True
        )[0].strip()

        hypotheses.append(hypothesis)
        references.append(reference)
        filenames.append(audio_file)

    except Exception as e:
        print(f"  Failed: {audio_file} — {e}")
        failed.append(audio_file)

print(f"\nProcessed: {len(hypotheses)} | Failed: {len(failed)}")

# ── Metrics ───────────────────────────────────────────────────────────────────

# 1. WER & CER
wer = jiwer.wer(references, hypotheses)
cer = jiwer.cer(references, hypotheses)

# 2. BLEU
bleu_result = BLEU().corpus_score(hypotheses, [references])

# 3. chrF++
chrf_result = CHRF(word_order=2).corpus_score(hypotheses, [references])

# 4. BERTScore
print("\nComputing BERTScore (takes ~1 min)...")

# Filter out any empty strings (causes crash)
valid_pairs = [(h, r) for h, r in zip(hypotheses, references) if h.strip() and r.strip()]
hyp_clean = [p[0] for p in valid_pairs]
ref_clean = [p[1] for p in valid_pairs]
print(f"  Valid pairs for BERTScore: {len(hyp_clean)} / {len(hypotheses)}")

try:
    P, R, F1 = bert_score(
        hyp_clean, ref_clean,
        model_type='bert-base-multilingual-cased',  # better for non-English
        verbose=False
    )
    bertscore_f1 = F1.mean().item()
    print(f"BERTScore F1: {bertscore_f1:.4f}")
except Exception as e:
    print(f"BERTScore failed: {e}")
    bertscore_f1 = 0.0
# ── Print Summary ─────────────────────────────────────────────────────────────
print("\n" + "="*45)
print("         EVALUATION RESULTS SUMMARY")
print("="*45)
print(f"  Total samples evaluated : {len(hypotheses)}")
print(f"  WER   (lower is better) : {wer*100:.2f}%")
print(f"  CER   (lower is better) : {cer*100:.2f}%")
print(f"  Word Accuracy           : {(1-wer)*100:.2f}%")
print(f"  BLEU  (higher=better)   : {bleu_result.score:.2f} / 100")
print(f"  chrF++ (higher=better)  : {chrf_result.score:.2f} / 100")
print(f"  BERTScore F1            : {bertscore_f1:.4f} / 1.0")
print("="*45)

# ── Sample Predictions ────────────────────────────────────────────────────────
print("\n── 10 Sample Predictions ──")
for i in range(min(10, len(hypotheses))):
    s_wer = jiwer.wer(references[i], hypotheses[i])
    print(f"\n[{i+1}] {filenames[i]}")
    print(f"  REF : {references[i]}")
    print(f"  HYP : {hypotheses[i]}")
    print(f"  WER : {s_wer*100:.1f}%")

# ── Save Results ──────────────────────────────────────────────────────────────
results_df = pd.DataFrame({
    'filename'  : filenames,
    'reference' : references,
    'hypothesis': hypotheses,
    'sample_wer': [round(jiwer.wer(r, h) * 100, 2) for r, h in zip(references, hypotheses)],
    'sample_cer': [round(jiwer.cer(r, h) * 100, 2) for r, h in zip(references, hypotheses)],
})
results_df.to_csv('/kaggle/working/evaluation_results.csv', index=False)

summary_df = pd.DataFrame([{
    'Total_Samples': len(hypotheses),
    'WER_%'        : round(wer * 100, 2),
    'CER_%'        : round(cer * 100, 2),
    'Word_Accuracy': round((1 - wer) * 100, 2),
    'BLEU'         : round(bleu_result.score, 2),
    'chrF++'       : round(chrf_result.score, 2),
    'BERTScore_F1' : round(bertscore_f1, 4),
}])
summary_df.to_csv('/kaggle/working/metrics_summary.csv', index=False)

print("\nSaved:")
print("  /kaggle/working/evaluation_results.csv  (per-sample)")
print("  /kaggle/working/metrics_summary.csv     (overall metrics)")

Device: cuda


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Model loaded successfully!
Matched pairs found: 244 / 244


Evaluating: 100%|██████████| 244/244 [01:34<00:00,  2.58it/s]



Processed: 244 | Failed: 0

Computing BERTScore (takes ~1 min)...
  Valid pairs for BERTScore: 243 / 244


...

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERTScore F1: 0.8260

         EVALUATION RESULTS SUMMARY
  Total samples evaluated : 244
  WER   (lower is better) : 75.38%
  CER   (lower is better) : 60.63%
  Word Accuracy           : 24.62%
  BLEU  (higher=better)   : 22.22 / 100
  chrF++ (higher=better)  : 37.60 / 100
  BERTScore F1            : 0.8260 / 1.0

── 10 Sample Predictions ──

[1] common_voice_bsk_41990073.wav
  REF : It was as long as a rope.
  HYP : It was a long, long time ago
  WER : 71.4%

[2] common_voice_bsk_41990074.wav
  REF : I am standing between Ammy and Mary
  HYP : I am standing between the two rivers.
  WER : 42.9%

[3] common_voice_bsk_41990075.wav
  REF : it is the story of Pegasus and a jinn
  HYP : It is the story of the Wind Horse and the Ghost.
  WER : 66.7%

[4] common_voice_bsk_41990076.wav
  REF : the wind has been blowing since yesterday
  HYP : It has been windy since yesterday.
  WER : 57.1%

[5] common_voice_bsk_41990077.wav
  REF : she put the mountain crepe on the griddle
  HYP : She is mo